# HDB resale — LightGBM v1 + SHAP

**Model:** `LGBMRegressor` on **`log1p(resale_price)`**; predictions transformed with **`expm1`** for RMSE in dollars.

**Validation:**
1. **Time-based:** last 12 calendar months holdout (`Tranc_YearMonth`).
2. **Random:** **70% / 30%** train–validation split (`train_test_split`, `shuffle=True`, `random_state=RNG`).

**SHAP:** `shap.TreeExplainer` on the **fitted random-split booster**, evaluated on a **random sample of the validation fold** (for speed).

**Dependencies:** `pip install lightgbm shap` or `conda install -c conda-forge lightgbm shap`.

**Engineering:** includes **`month_index`** = `(Tranc_Year - 2000) * 12 + Tranc_Month` (monotonic time index).

**Submissions:**
- `ROOT / submission / sub_lgb_v1_t.csv` (time split)
- `ROOT / submission / sub_lgb_v1_r.csv` (random split)


In [ ]:
# Paths
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd
TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUB_PATH = ROOT / "data" / "sample_sub_reg.csv"
SUBMISSION_PATH_T = ROOT / "submission" / "sub_lgb_v1_t.csv"
SUBMISSION_PATH_R = ROOT / "submission" / "sub_lgb_v1_r.csv"

RNG = 42

print(f"ROOT: {ROOT.resolve()}")
train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
print(train.shape, test.shape)


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

import lightgbm as lgb
import matplotlib.pyplot as plt
import shap

TARGET = "resale_price"



In [8]:
ROOMS_FROM_FLAT = {
    "1 ROOM": 1,
    "2 ROOM": 2,
    "3 ROOM": 3,
    "4 ROOM": 4,
    "5 ROOM": 5,
    "EXECUTIVE": 6,
    "MULTI-GENERATION": 7,
}


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    pc = out["postal"].astype(str).str.replace(r"\.0$", "", regex=True)
    out["postal_sector"] = pd.to_numeric(pc.str.slice(0, 2), errors="coerce")

    ms = pd.to_numeric(out["mid_storey"], errors="coerce")
    mx = pd.to_numeric(out["max_floor_lvl"], errors="coerce")
    out["storey_ratio"] = np.where(mx > 0, ms / mx, np.nan)

    rcols = ["1room_rental", "2room_rental", "3room_rental", "other_room_rental"]
    total_rent = np.zeros(len(out))
    for c in rcols:
        total_rent += pd.to_numeric(out[c], errors="coerce").fillna(0).to_numpy(dtype=float)
    td = pd.to_numeric(out["total_dwelling_units"], errors="coerce").to_numpy(dtype=float)
    out["rental_ratio"] = np.where(td > 0, total_rent / td, np.nan)

    out["rooms_num"] = out["flat_type"].map(ROOMS_FROM_FLAT).astype(float)
    ty = pd.to_numeric(out["Tranc_Year"], errors="coerce")
    tm = pd.to_numeric(out["Tranc_Month"], errors="coerce")
    out["month_index"] = (ty - 2000) * 12 + tm
    return out


train = add_engineered_features(train)
test = add_engineered_features(test)

DROP_FEATURES = [
    "id",
    "Tranc_YearMonth",
    "Tranc_Year",
    "Tranc_Month",
    "floor_area_sqft",
    "postal",
    "address",
    "block",
    "street_name",
    "flat_type",
    "flat_model",
    "1room_sold",
    "2room_sold",
    "3room_sold",
    "4room_sold",
    "5room_sold",
    "exec_sold",
    "multigen_sold",
    "studio_apartment_sold",
    "1room_rental",
    "2room_rental",
    "3room_rental",
    "other_room_rental",
    "bus_stop_name",
    "sec_sch_name",
]

feature_cols = [c for c in train.columns if c not in DROP_FEATURES and c != TARGET]
assert not set(feature_cols) - set(test.columns)

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y = train[TARGET].astype(float)

print(
    f"Features: {len(feature_cols)} (incl. month_index from Tranc_Year/Tranc_Month)"
)



Features: 56 (incl. month_index from Tranc_Year/Tranc_Month)


In [11]:
def imputation_stats(X_ref: pd.DataFrame, cat_cols: list, num_cols: list):
    """Train-only medians / modes for numeric and categorical columns."""
    num_med = {
        c: pd.to_numeric(X_ref[c], errors="coerce").median()
        for c in num_cols
    }
    cat_fill = {}
    for c in cat_cols:
        s = X_ref[c].astype(str).replace("nan", np.nan)
        m = s.mode(dropna=True)
        cat_fill[c] = m.iloc[0] if len(m) else "_MISSING_"
    return num_med, cat_fill


def prepare_for_lgb(
    X: pd.DataFrame,
    cat_cols: list,
    num_cols: list,
    num_med: dict,
    cat_fill: dict,
    cat_categories=None,
) -> pd.DataFrame:
    """Impute using train statistics; categoricals use fixed category levels when provided."""
    out = X.copy()
    for c in num_cols:
        v = pd.to_numeric(out[c], errors="coerce")
        out[c] = v.fillna(num_med[c])
    for c in cat_cols:
        s = out[c].astype(str).replace("nan", np.nan).fillna(cat_fill[c])
        if cat_categories is not None:
            allowed = set(cat_categories[c])
            s = s.where(s.isin(allowed), cat_fill[c])
            out[c] = pd.Categorical(s, categories=cat_categories[c])
        else:
            out[c] = s.astype("category")
    return out


num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]

X_tr_raw, X_va_raw, y_tr, y_va = train_test_split(
    X_train,
    y,
    test_size=0.20,
    random_state=RNG,
    shuffle=True,
)

num_med, cat_fill = imputation_stats(X_tr_raw, cat_cols, num_cols)
X_tr = prepare_for_lgb(X_tr_raw, cat_cols, num_cols, num_med, cat_fill, cat_categories=None)
cat_categories = {c: list(X_tr[c].cat.categories) for c in cat_cols}
X_va = prepare_for_lgb(X_va_raw, cat_cols, num_cols, num_med, cat_fill, cat_categories)

print(
    f"Random split 80/20: train {len(X_tr):,} rows ({100 * len(X_tr) / len(X_train):.1f}%), "
    f"val {len(X_va):,} rows ({100 * len(X_va) / len(X_train):.1f}%)"
)



Random split 80/20: train 120,507 rows (80.0%), val 30,127 rows (20.0%)


In [12]:
y_tr_log = np.log1p(y_tr.values)
y_va_log = np.log1p(y_va.values)

model = lgb.LGBMRegressor(
    objective="regression",
    random_state=RNG,
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    n_jobs=-1,
)

model.fit(
    X_tr,
    y_tr_log,
    eval_set=[(X_va, y_va_log)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)],
)

pred_va = np.expm1(model.predict(X_va, num_iteration=model.best_iteration_))
rmse = root_mean_squared_error(y_va, pred_va)
mae = mean_absolute_error(y_va, pred_va)
print(f"Validation RMSE (dollars): {rmse:,.2f}")
print(f"Validation MAE (dollars): {mae:,.2f}")
print(f"best_iteration_: {model.best_iteration_}")



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5355
[LightGBM] [Info] Number of data points in the train set: 120507, number of used features: 54
[LightGBM] [Info] Start training from score 12.968962
Validation RMSE (dollars): 21,532.05
Validation MAE (dollars): 15,553.81
best_iteration_: 1998


In [ ]:
# SHAP on a sample of the 30% validation holdout (TreeExplainer)
N_SHAP = min(4000, len(X_va))
rng = np.random.RandomState(RNG)
idx = rng.choice(X_va.index, size=N_SHAP, replace=False)
X_shap = X_va.loc[idx]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, show=False, max_display=25)
plt.title("SHAP summary (30% random validation holdout, sample)")
plt.tight_layout()
plt.show()



In [ ]:
# Refit split-specific full-train models and write _t / _r submissions
# Time split: derive best_iteration on time holdout, then refit full train with that tree count.
period = pd.to_datetime(train["Tranc_YearMonth"], format="%Y-%m")
cutoff = period.max() - pd.DateOffset(months=12)
tr_time = period < cutoff
va_time = ~tr_time
X_tr_raw_t = X_train.loc[tr_time]
X_va_raw_t = X_train.loc[va_time]
y_tr_t = y.loc[tr_time]
y_va_t = y.loc[va_time]
num_med_t, cat_fill_t = imputation_stats(X_tr_raw_t, cat_cols, num_cols)
X_tr_t = prepare_for_lgb(X_tr_raw_t, cat_cols, num_cols, num_med_t, cat_fill_t, cat_categories=None)
cat_categories_t = {c: list(X_tr_t[c].cat.categories) for c in cat_cols}
X_va_t = prepare_for_lgb(X_va_raw_t, cat_cols, num_cols, num_med_t, cat_fill_t, cat_categories_t)
y_tr_log_t = np.log1p(y_tr_t.values)
y_va_log_t = np.log1p(y_va_t.values)
model_time = lgb.LGBMRegressor(
    objective="regression",
    random_state=RNG,
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    n_jobs=-1,
)
model_time.fit(
    X_tr_t,
    y_tr_log_t,
    eval_set=[(X_va_t, y_va_log_t)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)],
)
num_med_full, cat_fill_full = imputation_stats(X_train, cat_cols, num_cols)
X_full = prepare_for_lgb(X_train, cat_cols, num_cols, num_med_full, cat_fill_full, cat_categories=None)
cat_categories_full = {c: list(X_full[c].cat.categories) for c in cat_cols}
X_test_p = prepare_for_lgb(X_test, cat_cols, num_cols, num_med_full, cat_fill_full, cat_categories_full)
y_log_full = np.log1p(y.values)
n_trees_t = int(model_time.best_iteration_) if getattr(model_time, "best_iteration_", None) is not None else 500
if n_trees_t < 50:
    n_trees_t = 500
n_trees_r = int(model.best_iteration_) if getattr(model, "best_iteration_", None) is not None else 500
if n_trees_r < 50:
    n_trees_r = 500
model_full_t = lgb.LGBMRegressor(
    objective="regression",
    random_state=RNG,
    n_estimators=n_trees_t,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    n_jobs=-1,
)
model_full_r = lgb.LGBMRegressor(
    objective="regression",
    random_state=RNG,
    n_estimators=n_trees_r,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    n_jobs=-1,
)
model_full_t.fit(X_full, y_log_full)
model_full_r.fit(X_full, y_log_full)
test_pred_t = np.expm1(model_full_t.predict(X_test_p))
test_pred_r = np.expm1(model_full_r.predict(X_test_p))
sample = pd.read_csv(SAMPLE_SUB_PATH, nrows=5)
sub_t = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_t})
sub_r = pd.DataFrame({"Id": test["id"], "Predicted": test_pred_r})
assert list(sub_t.columns) == list(sample.columns)
assert list(sub_r.columns) == list(sample.columns)
SUBMISSION_PATH_T.parent.mkdir(parents=True, exist_ok=True)
sub_t.to_csv(SUBMISSION_PATH_T, index=False)
sub_r.to_csv(SUBMISSION_PATH_R, index=False)
print(f"Final time model n_estimators: {n_trees_t}")
print(f"Final random model n_estimators: {n_trees_r}")
print(f"Wrote {SUBMISSION_PATH_T.resolve()} ({len(sub_t):,} rows)")
print(f"Wrote {SUBMISSION_PATH_R.resolve()} ({len(sub_r):,} rows)")
print(sub_t.head())


## Notes

- Numeric imputation uses **train-split (or full train) medians** only; categoricals use **train modes** and **train category levels** on val/test.
- SHAP summary reflects impact on the **log1p(price)** model output.
- Random 70/30 scores are often **optimistic** vs a **time-based** test; use time split when mimicking real forecasting.
- Adjust `N_SHAP` or `n_estimators` / early stopping for speed vs. fidelity.
